<a href="https://colab.research.google.com/github/vivianagonzalez203/Bootcamp-Data-Science/blob/main/02_Python/Clase_09_probabilidad_estadistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probabilidad y estadística para analítica de datos

Rápido es una empresa de domicilios con tres sucursales: Norte, Sur y Centro. Tenemos los 240 pedidos de septiembre y la gerencia quiere entender cómo le fue en el mes. En esta clase vamos a responder, con datos, las preguntas que haría cualquier gerente:

1. ¿Cuánto vale un pedido típico?
2. ¿Qué pedidos se salen de lo normal y qué sucursal es más predecible?
3. ¿Qué se relaciona con la calificación de los clientes?
4. ¿Qué tan probable es que un pedido llegue tarde y qué cambia ese riesgo?


## 1. Cargar y leer los datos

Ejecuta la celda. Si estás en Colab, aparece un botón para subir el archivo **domicilios.csv**.

In [1]:
import pandas as pd

try:
    df = pd.read_csv("domicilios.csv")
except FileNotFoundError:
    from google.colab import files
    files.upload()
    df = pd.read_csv("domicilios.csv")

df.head()

,id_pedido,fecha,sucursal,distancia_km,lluvia,tiempo_min,valor_pedido,calificacion
0,1001,2026-09-20,Sur,3.3,No,24,48100,4
1,1002,2026-09-12,Sur,5.5,Sí,34,42400,3
2,1003,2026-09-10,Norte,4.0,No,25,53100,5
3,1004,2026-09-28,Centro,7.2,Sí,44,17900,3
4,1005,2026-09-10,Sur,1.8,No,29,22400,5


| Columna | Qué contiene |
|---|---|
| `id_pedido` | Número del pedido |
| `fecha` | Día del pedido |
| `sucursal` | Norte, Sur o Centro |
| `distancia_km` | Distancia entre la sucursal y el cliente |
| `lluvia` | Si llovía durante la entrega: Sí o No |
| `tiempo_min` | Minutos que tardó la entrega |
| `valor_pedido` | Valor del pedido en pesos |
| `calificacion` | Calificación del cliente, de 1 a 5 |

**Ejemplo.** `describe()` calcula en una sola tabla las medidas que estudiaste en la plataforma.

In [2]:
df.describe().round(1)

,id_pedido,distancia_km,tiempo_min,valor_pedido,calificacion
count,240.0,240.0,240.0,240.0,240.0
mean,1120.5,5.0,35.0,55358.8,4.0
std,69.4,2.1,9.4,127885.1,0.7
min,1001.0,1.0,12.0,11600.0,2.0
25%,1060.8,3.4,29.0,31275.0,4.0
50%,1120.5,5.0,35.0,41000.0,4.0
75%,1180.2,6.6,41.0,62025.0,4.0
max,1240.0,8.9,72.0,2000000.0,5.0


**Cómo leer esta tabla.** Lee cada columna de arriba hacia abajo:

- **count** es la cantidad de pedidos: 240 en todas las columnas, así que no hay vacíos.
- En **tiempo_min**, la media (`mean`) es 35,0 y la mediana (`50%`) es 35. Están prácticamente iguales: los tiempos se reparten parejo alrededor del centro.
- En **valor_pedido**, la media es 55.358,8 y la mediana es 41.000. Están lejos una de la otra, y el máximo es de 2.000.000. Esa es la primera pista de que hay un valor extremo.
- En **calificacion**, el 25 %, el 50 % y el 75 % valen 4. La mayoría de clientes califica con 4.

**Ejercicio 1.** ¿Cuántos minutos tardó el pedido más demorado del mes? Completa con la función que da el valor máximo.

In [3]:
df["tiempo_min"].max()

72

Respuesta esperada: 72 minutos.

## 2. Media y mediana: ¿cuánto vale un pedido típico?

La media suma todos los valores y divide entre la cantidad. La mediana ordena los valores y toma el del centro. Cuando hay valores extremos, la media se mueve hacia ellos y la mediana casi no cambia.

**Ejemplo.** Calculamos las dos medidas del valor del pedido.

In [4]:
media = df["valor_pedido"].mean()
mediana = df["valor_pedido"].median()
print("Media:  ", round(media))
print("Mediana:", mediana)

Media:   55359
Mediana: 41000.0


La media está unos 14.000 pesos por encima de la mediana. Para saber por qué, miramos los pedidos más grandes.

**Ejemplo.** **sort_values** con **ascending=False** ordena de mayor a menor.

In [5]:
df.sort_values("valor_pedido", ascending=False).head(3)

,id_pedido,fecha,sucursal,distancia_km,lluvia,tiempo_min,valor_pedido,calificacion
57,1058,2026-09-12,Centro,3.2,No,25,2000000,5
69,1070,2026-09-01,Centro,6.1,No,37,126000,4
156,1157,2026-09-29,Centro,8.7,No,43,125400,4


El pedido 1058, de la sucursal Centro, vale 2.000.000. Fue un pedido corporativo para un evento. Veamos qué pasa con las medidas si lo dejamos por fuera.

**Ejemplo.** Filtramos los pedidos menores a 2.000.000 y volvemos a calcular.

In [6]:
sin_corporativo = df[df["valor_pedido"] < 2000000]
print("Media sin el corporativo:  ", round(sin_corporativo["valor_pedido"].mean()))
print("Mediana sin el corporativo:", sin_corporativo["valor_pedido"].median())

Media sin el corporativo:   47222
Mediana sin el corporativo: 40800.0


**Cómo leer el resultado.** Un solo pedido movía la media de 47.222 a 55.359 pesos. La mediana apenas cambió: de 40.800 a 41.000. Si la gerencia pregunta cuánto vale un pedido típico, la respuesta más honesta es la mediana: unos 41.000 pesos.

**Ejercicio 2.** Calcula la media y la mediana del tiempo de entrega. ¿Están cerca o lejos? ¿Qué te dice eso?

In [7]:
print("Media:  ", df["tiempo_min"].mean())
print("Mediana:", df["tiempo_min"].median())

Media:   35.0375
Mediana: 35.0


Respuesta esperada: media de 35,04 y mediana de 35. Están casi iguales, así que no hay valores extremos que muevan la media y cualquiera de las dos sirve como valor típico.

## 3. Cuartiles y valores atípicos

Los cuartiles parten los datos ordenados en cuatro grupos con la misma cantidad de datos. Q1 deja por debajo al 25 % de los datos, Q2 es la mediana y Q3 deja por debajo al 75 %.

En analítica los cuartiles sirven para tres cosas:

1. Ver dónde está la mitad central de los datos: entre Q1 y Q3.
2. Comparar grupos: un grupo con la mitad central más estrecha es más predecible.
3. Detectar valores atípicos: datos tan lejos del resto que vale la pena revisarlos.

**Ejemplo.** **quantile** recibe los percentiles como números entre 0 y 1.

In [8]:
q1 = df["tiempo_min"].quantile(0.25)
q2 = df["tiempo_min"].quantile(0.50)
q3 = df["tiempo_min"].quantile(0.75)
print("Q1:", q1, "| Q2 (mediana):", q2, "| Q3:", q3)

Q1: 29.0 | Q2 (mediana): 35.0 | Q3: 41.0


**Cómo leer el resultado.** El 25 % de los pedidos llega en 29 minutos o menos, la mitad en 35 o menos y el 75 % en 41 o menos. La mitad central de los pedidos tarda entre 29 y 41 minutos.

### Regla para encontrar valores atípicos

1. Calcula el rango intercuartílico: **IQR = Q3 - Q1**. Es el ancho de la mitad central.
2. Calcula los límites: el inferior es **Q1 - 1.5 * IQR** y el superior es **Q3 + 1.5 * IQR**.
3. Todo dato por fuera de esos límites es un posible atípico.

**Ejemplo.** Aplicamos la regla al tiempo de entrega.

In [9]:
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print("IQR:", iqr, "| Límite inferior:", limite_inferior, "| Límite superior:", limite_superior)

atipicos = df[(df["tiempo_min"] < limite_inferior) | (df["tiempo_min"] > limite_superior)]
atipicos

IQR: 12.0 | Límite inferior: 11.0 | Límite superior: 59.0


,id_pedido,fecha,sucursal,distancia_km,lluvia,tiempo_min,valor_pedido,calificacion
116,1117,2026-09-01,Sur,7.1,No,67,56700,3
175,1176,2026-09-11,Sur,7.5,No,72,65300,2


**Cómo leer el resultado.** Cualquier pedido de menos de 11 minutos o de más de 59 minutos se sale de lo normal. Solo hay dos, de 67 y 72 minutos.

Un valor atípico no se borra de forma automática. Primero se revisa: puede ser un error de digitación o un caso real que la gerencia necesita conocer. Aquí son pedidos reales que se demoraron mucho y conviene preguntar qué pasó con ellos.

**Ejercicio 3.** Aplica la misma regla al valor del pedido. Completa los espacios con el nombre de la columna y los percentiles.

In [10]:
q1_v = df["valor_pedido"].quantile(0.25)
q3_v = df["valor_pedido"].quantile(0.75)
iqr_v = q3_v - q1_v
limite_sup_v = q3_v + 1.5 * iqr_v
print("Límite superior:", limite_sup_v)

df[df["valor_pedido"] > limite_sup_v]

Límite superior: 108150.0


,id_pedido,fecha,sucursal,distancia_km,lluvia,tiempo_min,valor_pedido,calificacion
36,1037,2026-09-19,Norte,6.2,Sí,38,114000,4
57,1058,2026-09-12,Centro,3.2,No,25,2000000,5
69,1070,2026-09-01,Centro,6.1,No,37,126000,4
156,1157,2026-09-29,Centro,8.7,No,43,125400,4


> Respuesta esperada: el límite superior es 108.150 pesos y hay 4 pedidos por encima: 114.000, 125.400, 126.000 y el corporativo de 2.000.000. Los tres primeros son pedidos grandes pero posibles; el corporativo es el que distorsiona la media.

### Comparar grupos: ¿qué sucursal es más predecible?

Las tres sucursales tienen un tiempo promedio parecido. Pero un promedio igual no significa un servicio igual.

**Ejemplo.** Calculamos los cuartiles por sucursal con **groupby** y luego el rango intercuartílico de cada una.

In [11]:
cuartiles = df.groupby("sucursal")["tiempo_min"].quantile([0.25, 0.75]).unstack()
cuartiles["iqr"] = cuartiles[0.75] - cuartiles[0.25]
cuartiles

,0.25,0.75,iqr
sucursal,,,
Centro,30.0,41.0,11.0
Norte,30.5,38.5,8.0
Sur,27.0,43.0,16.0


**Cómo leer el resultado.** En Norte la mitad central de los pedidos se reparte en 8 minutos, en Centro en 11 y en Sur en 16. Sur es la sucursal menos predecible: el doble de ancho que Norte.

### Percentiles para fijar metas

Un percentil deja por debajo un porcentaje de los datos. Sirve para fijar metas basadas en datos, no en cifras inventadas.

**Ejemplo.** Mercadeo quiere premiar el 5 % de pedidos de mayor valor. Ese grupo empieza en el percentil 95.

In [12]:
umbral = df["valor_pedido"].quantile(0.95)
print("Umbral premium:", round(umbral))
print("Pedidos premium:", len(df[df["valor_pedido"] >= umbral]))

Umbral premium: 84955
Pedidos premium: 12


**Cómo leer el resultado.** Un pedido entra al programa premium desde unos 84.955 pesos. En septiembre hubo 12 pedidos así.

**Ejercicio 4.** Rápido quiere prometer un tiempo de entrega que se cumpla en 9 de cada 10 pedidos. ¿Qué percentil necesitas? Calcúlalo.

In [13]:
df["tiempo_min"].quantile(0.90)

np.float64(46.0)

> Respuesta esperada: el percentil 90 es 46 minutos. Rápido puede prometer entregas en 46 minutos o menos y cumplir en 9 de cada 10 pedidos.

## 4. Correlación: ¿qué se relaciona con la calificación?

La correlación mide si dos variables numéricas se mueven juntas. Va de -1 a 1: cerca de 1, cuando una sube la otra también; cerca de -1, cuando una sube la otra baja; cerca de 0, no hay relación lineal.

**Ejemplo.** **corr** calcula la correlación de todas las parejas de columnas numéricas. Quitamos **id_pedido** porque es solo un número de identificación.

In [14]:
df.drop(columns="id_pedido").corr(numeric_only=True).round(2)

,distancia_km,tiempo_min,valor_pedido,calificacion
distancia_km,1.00,0.52,-0.05,-0.30
tiempo_min,0.52,1.00,-0.08,-0.66
valor_pedido,-0.05,-0.08,1.00,0.11
calificacion,-0.30,-0.66,0.11,1.00


**Cómo leer la tabla.** Busca la fila **calificacion**:

- Con **tiempo_min** la correlación es -0,66: mientras más se demora un pedido, más baja la calificación. Es la relación más fuerte.
- Con **distancia_km** es -0,30. Sería fácil concluir que a los clientes lejanos no les gusta el servicio. Pero mira la correlación entre distancia y tiempo: 0,52. Los pedidos lejanos tardan más, y es la demora la que baja la calificación.

Por eso se dice que correlación no es causalidad: que dos variables se muevan juntas no prueba que una cause la otra.

**Ejercicio 5.** Calcula solo la correlación entre la distancia y el tiempo de entrega. Completa con el nombre de la segunda columna.

In [15]:
df["distancia_km"].corr(df["tiempo_min"])

np.float64(0.5202542686191001)

> Respuesta esperada: 0,52: una relación positiva moderada. A más distancia, más tiempo, aunque no es lo único que influye.

## 5. Probabilidad paso a paso

En analítica, la forma más práctica de estimar una probabilidad es contar: cuántas veces pasó algo, dividido entre el total de casos. La respuesta siempre es un número entre 0 y 1, que también se puede leer como porcentaje.

En toda esta sección vamos a decir que un pedido llegó tarde si tardó más de 45 minutos.

### 5.1 La probabilidad como proporción

**Ejemplo.** Una comparación crea una columna de True y False: True si el pedido llegó tarde.

In [16]:
tarde = df["tiempo_min"] > 45
tarde.head()

,tiempo_min
0,False
1,False
2,False
3,False
4,False


Ahora contamos los True con **sum()**, porque Python cuenta cada True como 1 y cada False como 0, y dividimos entre el total de pedidos.

In [17]:
pedidos_tarde = tarde.sum()
total = len(df)
print("Pedidos tarde:", pedidos_tarde, "de", total)
print("Probabilidad:", pedidos_tarde / total)

Pedidos tarde: 29 de 240
Probabilidad: 0.12083333333333333


Contar y dividir es lo mismo que sacar el promedio de los unos y ceros. Por eso **mean()** da la probabilidad directamente:

In [18]:
tarde.mean()

np.float64(0.12083333333333333)

**Cómo leer el resultado.** La probabilidad de que un pedido llegue tarde es 0,121: cerca de 12 de cada 100 pedidos.

**Ejercicio 6.** ¿Cuál es la probabilidad de que un cliente califique con 5?

In [19]:
cinco = df["calificacion"] == 5
cinco.mean()

np.float64(0.2125)

> Respuesta esperada: 0,2125: cerca de 21 de cada 100 clientes califica con 5.

### 5.2 El complemento

Un evento y su contrario siempre suman 1. Si la probabilidad de llegar tarde es 0,121, la de llegar a tiempo es 1 - 0,121.

**Ejemplo.** Lo calculamos de las dos formas para comprobar que da lo mismo.

In [20]:
print("Con el complemento:", 1 - tarde.mean())
print("Contando directo:  ", (df["tiempo_min"] <= 45).mean())

Con el complemento: 0.8791666666666667
Contando directo:   0.8791666666666667


**Ejercicio 7.** ¿Cuál es la probabilidad de que no llueva durante una entrega? Primero calcula la de que llueva y luego usa el complemento.

In [21]:
lluvia = df["lluvia"] == "Sí"
print("Llueve:   ", lluvia.mean())
print("No llueve:", 1 - lluvia.mean())

Llueve:    0.26666666666666666
No llueve: 0.7333333333333334


> Respuesta esperada: llueve en el 26,7 % de las entregas, así que no llueve en el 73,3 %.

### 5.3 Dos condiciones a la vez

Para que se cumplan dos condiciones al mismo tiempo se usa **&**, igual que en los filtros que ya conoces. Cada condición va entre paréntesis o ya guardada en una variable.

**Ejemplo.** ¿Qué tan probable es que un pedido llegue tarde y que además esté lloviendo?

In [22]:
lluvia = df["lluvia"] == "Sí"
(tarde & lluvia).mean()

np.float64(0.075)

**Cómo leer el resultado.** El 7,5 % de todos los pedidos llegó tarde y con lluvia: 18 de 240.

**Ejercicio 8.** ¿Qué tan probable es que un pedido llegue tarde y sea de la sucursal Sur?

In [23]:
sur = df["sucursal"] == "Sur"
(tarde & sur).mean()

np.float64(0.07083333333333333)

> Respuesta esperada: 0,071: 17 de los 240 pedidos fueron de Sur y llegaron tarde.

### 5.4 Probabilidad condicional

Hasta ahora miramos todos los pedidos. Pero la gerencia pregunta algo más preciso: si está lloviendo, ¿qué tan probable es llegar tarde? Eso es una probabilidad condicional. Se escribe P(tarde | lluvia) y se lee "probabilidad de llegar tarde dado que llueve".

La idea clave es que solo miras el grupo de la condición: los pedidos con lluvia.

**Ejemplo, forma 1: filtrar el grupo.** Nos quedamos con los pedidos con lluvia y calculamos la proporción de tarde dentro de ellos.

In [24]:
con_lluvia = df[df["lluvia"] == "Sí"]
print("Pedidos con lluvia:", len(con_lluvia))
(con_lluvia["tiempo_min"] > 45).mean()

Pedidos con lluvia: 64


np.float64(0.28125)

**Ejemplo, forma 2: la fórmula.** P(tarde | lluvia) = P(tarde y lluvia) ÷ P(lluvia). Da exactamente lo mismo.

In [25]:
(tarde & lluvia).mean() / lluvia.mean()

np.float64(0.28125)

**Cómo leer el resultado.** Cuando llueve, la probabilidad de llegar tarde es del 28,1 %: 18 de los 64 pedidos con lluvia. En todo el mes era del 12,1 %. La lluvia más que duplica el riesgo.

**Ejercicio 9.** ¿Cuánto cuesta llegar tarde? Calcula la probabilidad de una calificación de 3 o menos cuando el pedido llega tarde y cuando llega a tiempo.

In [26]:
llegaron_tarde = df[df["tiempo_min"] > 45]
llegaron_a_tiempo = df[df["tiempo_min"] <= 45]
print("Si llega tarde:   ", (llegaron_tarde["calificacion"] <= 3).mean())
print("Si llega a tiempo:", (llegaron_a_tiempo["calificacion"] <= 3).mean())

Si llega tarde:    0.7931034482758621
Si llega a tiempo: 0.12322274881516587


> Respuesta esperada: 79,3 % si llega tarde y 12,3 % si llega a tiempo. Llegar tarde multiplica por más de seis la probabilidad de una mala calificación.

### 5.5 Comparar todos los grupos con groupby

Cuando quieres la probabilidad condicional de varios grupos a la vez, **groupby** lo hace en una sola línea. Primero guardamos la condición como una columna del DataFrame.

**Ejemplo.** Probabilidad de llegar tarde con lluvia y sin lluvia.

In [27]:
df["tarde"] = df["tiempo_min"] > 45
df.groupby("lluvia")["tarde"].mean()

,tarde
lluvia,
No,0.06250
Sí,0.28125


**Cómo leer el resultado.** Cada fila es una probabilidad condicional: sin lluvia, 6,2 %; con lluvia, 28,1 %.

### 5.6 Independencia

Dos eventos son independientes si saber que ocurrió uno no cambia la probabilidad del otro. Para comprobarlo, comparas la probabilidad condicional con la general:

- P(tarde) = 12,1 %
- P(tarde | lluvia) = 28,1 %

Como son muy distintas, llegar tarde y llover no son independientes: la lluvia sí cambia el riesgo. Si fueran independientes, las dos probabilidades serían parecidas.

**Ejercicio 10.** ¿Llegar tarde depende de la sucursal? Calcula la probabilidad de llegar tarde en cada sucursal y compárala con la general (12,1 %).

In [28]:
df.groupby("sucursal")["tarde"].mean()

,tarde
sucursal,
Centro,0.098765
Norte,0.050633
Sur,0.212500


Respuesta esperada: Norte 5,1 %, Centro 9,9 % y Sur 21,2 %. La de Sur está muy por encima del 12,1 % general, así que llegar tarde no es independiente de la sucursal.

## 6. Cierre: recomendaciones para la gerencia

Un número sin interpretación no le sirve a nadie. Con lo que calculamos en la clase, escribe tres recomendaciones para la gerencia de Rápido. Cada una debe apoyarse en un número.

 Ejemplo de recomendaciones:
1. Reportar el valor típico del pedido con la mediana, unos 41.000 pesos, y no con la media, que infla un pedido corporativo.
2. Revisar la operación de la sucursal Sur: su rango intercuartílico es el doble del de Norte y su probabilidad de llegar tarde es del 21,2 %, frente al 12,1 % general.
3. Reforzar repartidores los días de lluvia: el riesgo de llegar tarde pasa del 6,2 % al 28,1 %, y un pedido tarde tiene un 79 % de probabilidad de recibir 3 o menos.

## Actividad

Son tres puntos cortos que repiten lo que hiciste en clase. Resuélvelos en este mismo cuaderno y entrégalo por el medio que indique tu docente.

**Punto 1.** Aplica la regla de los cuartiles a **distancia_km**. ¿Hay pedidos atípicos por distancia? Escribe tu conclusión en una línea.

In [29]:
q1_d = df["distancia_km"].quantile(0.25)
q3_d = df["distancia_km"].quantile(0.75)
iqr_d = q3_d - q1_d
print("Límites:", q1_d - 1.5 * iqr_d, "y", q3_d + 1.5 * iqr_d)
df[(df["distancia_km"] < q1_d - 1.5 * iqr_d) | (df["distancia_km"] > q3_d + 1.5 * iqr_d)]

Límites: -1.5 y 11.5


,id_pedido,fecha,sucursal,distancia_km,lluvia,tiempo_min,valor_pedido,calificacion,tarde


Respuesta esperada: los límites son -1,5 y 11,5 km y ningún pedido queda por fuera. No hay distancias atípicas: todos los pedidos están dentro del rango normal de entrega. Que no aparezcan atípicos también es una conclusión válida.

**Punto 2.** ¿La lluvia cambia la probabilidad de recibir una calificación de 3 o menos? Calcula P(calificación de 3 o menos) en todos los pedidos y P(calificación de 3 o menos | lluvia). Compáralas y di si los eventos son independientes.

In [30]:
print("General:   ", (df["calificacion"] <= 3).mean())
con_lluvia = df[df["lluvia"] == "Sí"]
print("Con lluvia:", (con_lluvia["calificacion"] <= 3).mean())

General:    0.20416666666666666
Con lluvia: 0.390625


Respuesta esperada: 20,4 % en general y 39,1 % con lluvia. Con lluvia la probabilidad casi se duplica, así que no son independientes. Se conecta con lo visto en clase: la lluvia alarga las entregas y las demoras bajan la calificación.

**Punto 3.** Escribe una recomendación de una frase para la gerencia, apoyada en uno de los números de los puntos 1 o 2.

Ejemplo: en los días de lluvia, avisar al cliente que el pedido puede demorarse, porque la probabilidad de una mala calificación pasa del 20 % al 39 %.